In [ ]:
import pandas as pd

In [ ]:
year = snakemake.wildcards.year
date_range = snakemake.params.date_range
date_range = [year + "-" + date for date in date_range]
date_range = pd.date_range(date_range[0], date_range[1] + ' 23', freq='h')

In [ ]:
aggregated_regions = snakemake.params.aggregated_regions
aggregated_regions

In [ ]:
# connected_vehicles.tsv
connected_vehicles = pd.read_csv(snakemake.input[0]) #, sep='\t', header=None, index_col=0)
connected_vehicles.charging_cap.iloc[:len(date_range)].round(3).div(1000).to_csv(snakemake.output[0], sep = '\t', header = False)

In [ ]:
# demand_driving_MWh.tsv
demand_driving = pd.read_csv(snakemake.input[1]) #, sep='\t', header=None, index_col=0)
demand_driving.set_index(demand_driving.reset_index()["index"].astype('str') + '.EV')['battery discharge kWh'].iloc[:len(date_range)].div(1000).to_csv(snakemake.output[1], sep = '\t', header = False)

In [ ]:
# demand_ev_charging_MWh.tsv
demand_ev_charging = pd.read_csv(snakemake.input[2]) #, sep='\t', header=None, index_col=0)
demand_ev_charging.charge_battery.iloc[:len(date_range)].round(6).div(1000).to_csv(snakemake.output[2], sep = '\t', header = False)

In [ ]:
# ev_scalars.tsv
import shutil
shutil.copy2(snakemake.input[3], snakemake.output[3])

In [ ]:
# vehicles_zones.tsv
vehicles_zones = pd.read_csv(snakemake.input[4], sep='\t')
vehicles_zones[['freq', 'unit', 'age', 'geo']] = vehicles_zones["freq,unit,age,geo\\TIME_PERIOD"].str.split(',', expand=True)
vehicles_zones.columns = vehicles_zones.columns.str.strip()
vehicles_zones = vehicles_zones.query("age == 'TOTAL'").set_index('geo').rename(index={'EL':'GR'})['2023']
vehicles_zones[aggregated_regions].to_csv(snakemake.output[4], sep='\t', header=False)

In [ ]:
# # temperature
# import atlite
# cutout = atlite.Cutout(path=snakemake.input.weatherdata)
# cutout.prepare(features=["temperature"])
# 
# import geopandas as gpd
# euroshape = gpd.read_file(snakemake.input.euroshape).set_index('index')
# temperature = cutout.temperature(shapes=euroshape, per_unit=True)
# 
# df_temperature = temperature.to_dataframe(name='temperature')
# df_temperature = pd.pivot(df_temperature.reset_index(), index='time', columns='index', values='temperature')
# df_temperature.resample('ME').mean()
# 
# ### look into availability matrix for an alternative way to calculate the mean temperature.